# BERT 양방향 사전학습 실습 - 추가 실습 코드: Masked Language Model 마스킹 구현

- Tutorial ID: `mod-bert-mlm-lab`
- Tutorial: BERT 양방향 사전학습 실습
- Section ID: `practice-mlm-masking`
- Section: 추가 실습 코드: Masked Language Model 마스킹 구현


# Integrated Notebook: 06_MLM_Masking_Implementation

## Original Version
Source: 068_mod-bert-mlm-lab_practice-mlm-masking_BERT_#Uc591#Ubc29#Ud5a5_#Uc0ac#Uc804#Ud559#Uc2b5_#Uc2e4#Uc2b5_-_#Ucd94#Uac00_#Uc2e4#Uc2b5_#Ucf54#Ub4dc-_Masked_Language_Model_#Ub9c8#Uc2a4#Ud0b9_#Uad6c#Ud604.ipynb

# BERT 양방향 사전학습 실습 - 추가 실습 코드: Masked Language Model 마스킹 구현

- Tutorial ID: `mod-bert-mlm-lab`
- Tutorial: BERT 양방향 사전학습 실습
- Section: 추가 실습 코드: Masked Language Model 마스킹 구현

In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Masked Language Model 마스킹 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def create_mlm_data(tokens, mask_token="[MASK]", vocab_size=30000, mask_prob=0.15):
    """BERT MLM 마스킹: 80% MASK, 10% Random, 10% Keep"""
        tokens = list(tokens)
    n = len(tokens)
    labels = [None] * n
    
        num_to_mask = int(n * mask_prob)
        mask_indices = np.random.choice(n, num_to_mask, replace=False)
    
    print(f"총 {n}개 토큰 중 {num_to_mask}개 마스킹 ({mask_prob*100:.0f}%)\\n")
    
        for idx in mask_indices:
        labels[idx] = tokens[idx]
        rand = np.random.random()
        
        if rand < 0.8:
            tokens[idx] = mask_token
            action = "→ [MASK]"
        elif rand < 0.9:
            tokens[idx] = f"[RAND]"
            action = "→ [RANDOM]"
        else:
            action = "(유지)"
        
        print(f"  위치 {idx:2d}: '{labels[idx]:10s}' {action}")
    
    return tokens, labels

np.random.seed(42)
original = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog", "today"]

print("=" * 50)
print("BERT Masked Language Model (MLM)")
print("=" * 50)
print(f"\\n원본: {' '.join(original)}\\n")

masked_tokens, labels = create_mlm_data(original)

print(f"\\n결과: {' '.join(masked_tokens)}")
print(f"\\n📊 80/10/10 전략의 이유:")
print("  - 80% [MASK]: 양방향 문맥에서 예측 학습")
print("  - 10% Random: 노이즈로 robust한 표현 학습")
print("  - 10% Keep: Fine-tuning 시 [MASK] 없어도 동작")

## AI Developed Version 1
Source: 068_Masked_Language_Model_#Ub9c8#Uc2a4#Ud0b9_#Uad6c#Ud604.ipynb

# 📘 Masked Language Model 마스킹 구현 - 추가 실습

## BERT 양방향 사전학습 실습 - 심화편

---

### 🎯 학습 목표

1. 텐서 연산을 활용한 효율적인 마스킹 구현
2. Whole Word Masking (WWM) 이해 및 구현
3. 마스킹 통계 분석 및 시각화
4. 실제 BERT 모델로 마스킹된 토큰 예측해보기

### 📋 선수 지식

- 노트북 067의 MLM 기본 개념
- BERT 마스킹 전략 (80/10/10 규칙)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch 버전: {torch.__version__}")

try:
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("✅ transformers 라이브러리 로드 완료")
except ImportError:
    print("❌ !pip install transformers")

## 실습 1: 텐서 기반 효율적 마스킹

이전 노트북에서는 for 루프로 마스킹을 구현했습니다.
이번에는 **텐서 연산**으로 훨씬 빠르게 구현합니다.

### 핵심 아이디어

1. `torch.bernoulli`로 마스킹 위치 한 번에 결정
2. `torch.where`로 조건부 토큰 교체
3. 반복문 없이 배치 전체를 한 번에 처리

In [ ]:
# ============================================================
# 텐서 기반 MLM 마스킹 구현

In [ ]:
def tensor_mlm_masking(input_ids, tokenizer, mlm_probability=0.15):
    """
    텐서 연산을 활용한 효율적인 MLM 마스킹.
    
    for 루프 없이 배치 전체를 한 번에 마스킹합니다.
    
    Args:
        input_ids: (batch_size, seq_len) 형태의 텐서
        tokenizer: 토크나이저
        mlm_probability: 마스킹 확률
    
    Returns:
        masked_input_ids: 마스킹된 입력
        labels: 정답 레이블 (-100 = 무시)
    """
    # 원본 복사
    labels = input_ids.clone()
    
    # Step 1: 특수 토큰 위치 파악 (마스킹 금지 영역)
    # 특수 토큰: [CLS], [SEP], [PAD]
    special_tokens_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for special_id in [tokenizer.cls_token_id, tokenizer.sep_token_id, 
                       tokenizer.pad_token_id]:
        special_tokens_mask |= (input_ids == special_id)
    
    # Step 2: 마스킹 대상 선택
    # torch.bernoulli: 각 위치에 대해 확률적으로 True/False 생성
    # full_like: input_ids와 같은 shape의 텐서를 mlm_probability로 채움
    probability_matrix = torch.full_like(input_ids, mlm_probability, dtype=torch.float)
    probability_matrix.masked_fill_(special_tokens_mask, 0.0)  # 특수 토큰은 확률 0
    
    # 베르누이 분포로 마스킹 위치 결정
    masked_indices = torch.bernoulli(probability_matrix).bool()
    
    # Step 3: 레이블 설정
    # 마스킹되지 않은 위치는 -100 (loss 무시)
    labels[~masked_indices] = -100
    
    # Step 4: 80% → [MASK]로 교체
    indices_replaced = torch.bernoulli(
        torch.full_like(input_ids, 0.8, dtype=torch.float)
    ).bool() & masked_indices
    input_ids[indices_replaced] = tokenizer.mask_token_id
    
    # Step 5: 10% → 랜덤 토큰으로 교체
    # (나머지 20% 중 절반 = 전체의 10%)
    indices_random = torch.bernoulli(
        torch.full_like(input_ids, 0.5, dtype=torch.float)
    ).bool() & masked_indices & ~indices_replaced
    random_words = torch.randint(
        0, tokenizer.vocab_size, input_ids.shape, dtype=input_ids.dtype
    )
    input_ids[indices_random] = random_words[indices_random]
    
    # 나머지 10%는 원래 토큰 유지 (아무것도 안 함)
    
    return input_ids, labels


# 테스트
sentences = [
    "The quick brown fox jumps over the lazy dog",
    "I enjoy learning about natural language processing",
    "Deep learning has revolutionized artificial intelligence",
]

# 배치 토큰화
encoded = tokenizer(
    sentences, 
    padding=True,           # 패딩 추가
    return_tensors="pt"     # PyTorch 텐서 반환
)

print("원본 input_ids shape:", encoded['input_ids'].shape)
print()

# 마스킹 적용
input_ids_copy = encoded['input_ids'].clone()
masked_ids, labels = tensor_mlm_masking(input_ids_copy, tokenizer)

# 결과 확인
for i, sent in enumerate(sentences):
    original_tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][i])
    masked_tokens = tokenizer.convert_ids_to_tokens(masked_ids[i])
    
    print(f"문장 {i+1}: {sent}")
    print(f"  원본:  {original_tokens}")
    print(f"  마스킹: {masked_tokens}")
    
    changes = [(j, original_tokens[j], masked_tokens[j]) 
               for j in range(len(original_tokens)) 
               if labels[i, j] != -100]
    if changes:
        for pos, orig, masked in changes:
            print(f"    위치 {pos}: '{orig}' → '{masked}'")
    print()

## 실습 2: Whole Word Masking (WWM)

### 문제: 서브워드 마스킹의 한계

BERT 토크나이저는 단어를 서브워드로 나눕니다:
```
"playing" → ["play", "##ing"]
```

기본 마스킹에서 "##ing"만 마스킹되면:
```
"I am play [MASK]" → 너무 쉬움! ("##ing"은 거의 확정적)
```

### 해결: Whole Word Masking

단어의 일부가 마스킹되면, 같은 단어의 **모든 서브토큰**을 함께 마스킹합니다:
```
"I am [MASK] [MASK]" → "play" + "##ing" 전체 마스킹
```

In [ ]:
# ============================================================
# Whole Word Masking 구현

In [ ]:
def whole_word_masking(input_ids, tokenizer, mlm_probability=0.15):
    """
    Whole Word Masking을 구현합니다.
    
    서브워드(##로 시작하는 토큰)가 마스킹되면,
    해당 단어의 모든 서브토큰을 함께 마스킹합니다.
    """
    labels = input_ids.clone()
    batch_size, seq_len = input_ids.shape
    
    for batch_idx in range(batch_size):
        tokens = tokenizer.convert_ids_to_tokens(
            input_ids[batch_idx].tolist()
        )
        
        # 단어 그룹 만들기
        # 예: ["play", "##ing", "with", "cat", "##s"]
        # → 그룹: [[0], [1], [2], [3, 4]]... 아니, 아래처럼:
        # → word_groups: [["play", "##ing"], ["with"], ["cat", "##s"]]
        # → word_group_indices: [[0, 1], [2], [3, 4]]
        word_group_indices = []  # 각 단어의 토큰 인덱스들
        current_group = []
        
        for i, token in enumerate(tokens):
            # 특수 토큰 건너뛰기
            if token in [tokenizer.cls_token, tokenizer.sep_token, 
                         tokenizer.pad_token]:
                if current_group:
                    word_group_indices.append(current_group)
                    current_group = []
                continue
            
            if token.startswith("##"):
                # 서브워드: 이전 단어 그룹에 추가
                current_group.append(i)
            else:
                # 새 단어 시작
                if current_group:
                    word_group_indices.append(current_group)
                current_group = [i]
        
        if current_group:
            word_group_indices.append(current_group)
        
        # 단어 단위로 마스킹 결정
        masked_positions = set()
        for group in word_group_indices:
            if np.random.random() < mlm_probability:
                # 이 단어의 모든 서브토큰을 마스킹
                for idx in group:
                    masked_positions.add(idx)
        
        # 마스킹 적용
        for i in range(seq_len):
            if i not in masked_positions:
                labels[batch_idx, i] = -100  # 예측 대상 아님
            else:
                # 80/10/10 규칙 적용
                rand = np.random.random()
                if rand < 0.8:
                    input_ids[batch_idx, i] = tokenizer.mask_token_id
                elif rand < 0.9:
                    input_ids[batch_idx, i] = np.random.randint(
                        0, tokenizer.vocab_size
                    )
                # else: 유지
    
    return input_ids, labels


# 테스트 - 서브워드가 있는 문장
test_sentences = [
    "The preprocessing step involves tokenizing and embedding",
    "Transformers are revolutionizing natural language understanding",
]

encoded_wwm = tokenizer(test_sentences, padding=True, return_tensors="pt")

print("=== Whole Word Masking 테스트 ===")
print()

for i, sent in enumerate(test_sentences):
    tokens = tokenizer.convert_ids_to_tokens(encoded_wwm['input_ids'][i])
    print(f"문장 {i+1}: {sent}")
    print(f"  토큰: {tokens}")
    
    # 서브워드 확인
    subwords = [t for t in tokens if t.startswith('##')]
    if subwords:
        print(f"  서브워드: {subwords}")
    print()

# WWM 적용
ids_copy = encoded_wwm['input_ids'].clone()
masked_wwm, labels_wwm = whole_word_masking(ids_copy, tokenizer)

for i in range(len(test_sentences)):
    orig_tokens = tokenizer.convert_ids_to_tokens(encoded_wwm['input_ids'][i])
    mask_tokens = tokenizer.convert_ids_to_tokens(masked_wwm[i])
    
    print(f"문장 {i+1} 마스킹 결과:")
    print(f"  원본:  {orig_tokens}")
    print(f"  마스킹: {mask_tokens}")
    print()

## 실습 3: 마스킹 통계 분석

마스킹이 실제로 15% 비율로 잘 적용되는지,
80/10/10 비율이 맞는지 통계적으로 확인해봅시다.

In [ ]:
# ============================================================
# 마스킹 통계 분석

In [ ]:
# 많은 반복으로 통계 수집
num_trials = 1000
test_text = "The quick brown fox jumps over the lazy dog near the river"
encoded_stats = tokenizer(test_text, return_tensors="pt")
original_ids = encoded_stats['input_ids']

total_tokens = 0       # 전체 일반 토큰 수
total_masked = 0       # 마스킹된 토큰 수
count_mask_token = 0   # [MASK]로 교체된 수
count_random = 0       # 랜덤 토큰으로 교체된 수
count_kept = 0         # 원래 유지된 수

# 특수 토큰 제외한 일반 토큰 수
special_ids = {tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id}
regular_tokens = sum(1 for id in original_ids[0].tolist() if id not in special_ids)

for _ in range(num_trials):
    ids_copy = original_ids.clone()
    masked_ids, labels = tensor_mlm_masking(ids_copy, tokenizer, 0.15)
    
    # 마스킹된 위치 (labels != -100)
    masked_positions = (labels != -100)
    num_masked = masked_positions.sum().item()
    
    total_tokens += regular_tokens
    total_masked += num_masked
    
    for i in range(labels.shape[1]):
        if labels[0, i] != -100:
            if masked_ids[0, i] == tokenizer.mask_token_id:
                count_mask_token += 1
            elif masked_ids[0, i] == original_ids[0, i]:
                count_kept += 1
            else:
                count_random += 1

print(f"=== 마스킹 통계 ({num_trials}회 시행) ===")
print()
print(f"전체 일반 토큰 수: {total_tokens}")
print(f"마스킹된 토큰 수: {total_masked}")
print(f"마스킹 비율: {total_masked/total_tokens*100:.2f}% (목표: 15%)")
print()
print(f"마스킹된 토큰 중:")
print(f"  [MASK]로 교체: {count_mask_token} ({count_mask_token/total_masked*100:.1f}%, 목표: 80%)")
print(f"  랜덤 교체:     {count_random} ({count_random/total_masked*100:.1f}%, 목표: 10%)")
print(f"  원래 유지:     {count_kept} ({count_kept/total_masked*100:.1f}%, 목표: 10%)")

In [ ]:
# ============================================================
# 마스킹 통계 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 파이 차트 1: 마스킹 vs 비마스킹
ax1 = axes[0]
ax1.pie(
    [total_masked, total_tokens - total_masked],
    labels=[f'마스킹\n({total_masked/total_tokens*100:.1f}%)', 
            f'유지\n({(total_tokens-total_masked)/total_tokens*100:.1f}%)'],
    colors=['#FF6B6B', '#4ECDC4'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11}
)
ax1.set_title('마스킹 비율 (목표: 15%)', fontsize=13)

# 파이 차트 2: 마스킹 방법 비율
ax2 = axes[1]
ax2.pie(
    [count_mask_token, count_random, count_kept],
    labels=[f'[MASK]\n({count_mask_token/total_masked*100:.1f}%)',
            f'랜덤 교체\n({count_random/total_masked*100:.1f}%)',
            f'원래 유지\n({count_kept/total_masked*100:.1f}%)'],
    colors=['#FF6B6B', '#FFE66D', '#4ECDC4'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11}
)
ax2.set_title('마스킹 방법 비율 (목표: 80/10/10)', fontsize=13)

plt.tight_layout()
plt.show()

## 실습 4: 실제 BERT로 마스킹된 토큰 예측하기

학습된 BERT 모델을 사용하여
[MASK] 위치의 토큰을 실제로 예측해봅시다!

In [ ]:
# ============================================================
# BERT 모델로 [MASK] 예측

In [ ]:
# 학습된 BERT 모델 로드
model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
model.eval()  # 평가 모드

print(f"BERT 모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print()

# 테스트 문장들
test_cases = [
    "The cat sat on the [MASK].",
    "I want to [MASK] a book.",
    "Paris is the capital of [MASK].",
    "She is a [MASK] at the hospital.",
    "The [MASK] is shining brightly today.",
]

print("=== BERT의 [MASK] 토큰 예측 ===")
print()

for text in test_cases:
    # 토큰화
    inputs = tokenizer(text, return_tensors="pt")
    
    # [MASK] 위치 찾기
    mask_positions = (inputs['input_ids'] == tokenizer.mask_token_id).nonzero()
    
    # 모델 예측
    with torch.no_grad():
        outputs = model(**inputs)
    
    # 예측 결과에서 상위 5개 토큰 추출
    for pos in mask_positions:
        mask_idx = pos[1].item()
        logits = outputs.logits[0, mask_idx]
        probs = F.softmax(logits, dim=-1)
        top5_probs, top5_indices = probs.topk(5)
        
        print(f"문장: '{text}'")
        print(f"  [MASK] 위치의 Top-5 예측:")
        for rank, (prob, idx) in enumerate(zip(top5_probs, top5_indices)):
            token = tokenizer.decode([idx])
            filled = text.replace('[MASK]', f'**{token}**', 1)
            print(f"    {rank+1}. '{token}' (확률: {prob:.4f}) → {filled}")
        print()

In [ ]:
# ============================================================
# 여러 [MASK]가 있는 경우

In [ ]:
text_multi = "The [MASK] [MASK] is very [MASK] today."
inputs_multi = tokenizer(text_multi, return_tensors="pt")

with torch.no_grad():
    outputs_multi = model(**inputs_multi)

mask_positions_multi = (inputs_multi['input_ids'] == tokenizer.mask_token_id).nonzero()

print(f"문장: '{text_multi}'")
print(f"[MASK] 위치: {[p[1].item() for p in mask_positions_multi]}")
print()

predicted_tokens = []
for pos in mask_positions_multi:
    mask_idx = pos[1].item()
    logits = outputs_multi.logits[0, mask_idx]
    top_token_id = logits.argmax().item()
    top_token = tokenizer.decode([top_token_id])
    predicted_tokens.append(top_token)
    
    probs = F.softmax(logits, dim=-1)
    top3_probs, top3_ids = probs.topk(3)
    top3_tokens = [tokenizer.decode([id]) for id in top3_ids]
    print(f"  [MASK] at position {mask_idx}: Top-3 = {list(zip(top3_tokens, top3_probs.tolist()))}")

result = text_multi
for token in predicted_tokens:
    result = result.replace('[MASK]', token, 1)
print(f"\n완성된 문장: '{result}'")

## 실습 5: 마스킹 위치에 따른 예측 난이도

문장에서 어떤 위치의 토큰이 예측하기 쉽고 어려운지 분석합니다.

In [ ]:
# ============================================================
# 각 위치별 예측 난이도 분석

In [ ]:
analysis_text = "The student went to the library to study for the exam"
encoded_analysis = tokenizer(analysis_text, return_tensors="pt")
tokens_analysis = tokenizer.convert_ids_to_tokens(encoded_analysis['input_ids'][0])

print(f"분석 문장: '{analysis_text}'")
print(f"토큰: {tokens_analysis}")
print()

# 각 위치를 하나씩 마스킹하고 예측 정확도 확인
print(f"{'위치':>4} {'토큰':>10} {'Top-1 예측':>10} {'확률':>8} {'정답?':>5}")
print("-" * 45)

difficulties = []

for i in range(1, len(tokens_analysis) - 1):  # [CLS]와 [SEP] 제외
    # i 위치만 마스킹
    masked_input = encoded_analysis['input_ids'].clone()
    original_token_id = masked_input[0, i].item()
    masked_input[0, i] = tokenizer.mask_token_id
    
    with torch.no_grad():
        outputs = model(input_ids=masked_input)
    
    logits = outputs.logits[0, i]
    probs = F.softmax(logits, dim=-1)
    top_prob, top_id = probs.topk(1)
    
    predicted_token = tokenizer.decode([top_id[0]])
    original_token = tokens_analysis[i]
    correct = "✅" if top_id[0].item() == original_token_id else "❌"
    correct_prob = probs[original_token_id].item()
    
    difficulties.append((original_token, correct_prob))
    
    print(f"{i:4d} {original_token:>10} {predicted_token:>10} {top_prob[0]:.4f}   {correct}")

print()
print("[해석]")
print("  확률이 높을수록 → 문맥에서 예측하기 쉬운 단어")
print("  확률이 낮을수록 → 문맥에서 예측하기 어려운 단어 (더 많은 정보를 담고 있음)")

## 🔑 핵심 정리

### 이번 노트북에서 배운 것

1. **텐서 기반 마스킹**: for 루프 없이 효율적으로 마스킹 구현
2. **Whole Word Masking**: 서브워드를 단어 단위로 함께 마스킹
3. **통계 검증**: 15% 비율과 80/10/10 규칙이 잘 적용되는지 확인
4. **실제 예측**: 학습된 BERT가 [MASK]를 얼마나 잘 맞추는지 실험

### MLM 마스킹 구현 체크리스트

- [ ] 특수 토큰([CLS], [SEP], [PAD]) 마스킹 제외
- [ ] 15% 확률로 마스킹 대상 선택
- [ ] 80% [MASK] / 10% 랜덤 / 10% 유지
- [ ] labels에서 비마스킹 위치는 -100
- [ ] (선택) Whole Word Masking 적용

### 전체 시리즈 요약

| 노트북 | 주제 | 핵심 |
|--------|------|------|
| 063 | Scaled Dot-Product Attention | Q, K, V 개념, 수식 구현 |
| 064 | Attention 심화 | nn.Linear, Padding Mask |
| 065 | Causal Attention | 하삼각 마스크, Auto-regressive |
| 066 | Multi-Head Attention | Head 분리/합치기, Cross-Attention |
| 067 | MLM Data Collator | BERT 마스킹 전략, Data Collator |
| 068 | MLM 마스킹 심화 | WWM, 통계 검증, BERT 예측 |

## AI Developed Version 2
Source: 06_MLM_Masking_Advanced.ipynb

# 📚 Notebook 6: Masked Language Model 마스킹 심화 구현

## BERT 마스킹 전략 심화 + 고급 기법

---

### 🎯 학습 목표
BERT의 마스킹 전략을 더 깊이 이해하고, 다양한 마스킹 기법을 구현합니다.

### 📋 목차
1. 기본 마스킹 vs Whole Word Masking (WWM)
2. Dynamic Masking vs Static Masking
3. 마스킹 비율에 따른 학습 효과 실험
4. SpanBERT 스타일 연속 마스킹
5. HuggingFace Transformers 활용 (실전)

### ⚠️ 사전 지식
- Notebook 5 (BERT MLM Data Collator) 완료 필수

---

In [ ]:
# ============================================================
# 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import copy
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

print("✅ 라이브러리 로딩 완료!")

In [ ]:
# ============================================================
# Notebook 5의 토크나이저와 기본 마스킹 함수 (복습)

In [ ]:
class SimpleTokenizer:
    """간단한 BERT 스타일 토크나이저"""
    def __init__(self):
        self.special_tokens = {
            '[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4,
        }
        words = [
            '나는', '너는', '우리', '그는', '그녀는',
            '고양이', '강아지', '새', '물고기', '토끼',
            '를', '을', '이', '가', '에서', '에', '와', '과',
            '좋아한다', '싫어한다', '키운다', '먹는다', '본다',
            '집', '학교', '공원', '도서관', '병원',
            '어제', '오늘', '내일', '항상', '가끔',
            '크다', '작다', '예쁘다', '귀엽다', '빠르다',
            '읽다', '쓰다', '달리다', '걷다', '잠을잔다',
            '##이', '##를', '##가', '##에서', '##는',  # 서브워드 토큰
        ]
        self.token2id = {**self.special_tokens}
        for i, word in enumerate(words):
            self.token2id[word] = len(self.special_tokens) + i
        self.id2token = {v: k for k, v in self.token2id.items()}
        self.vocab_size = len(self.token2id)
        self.pad_token_id = 0
        self.mask_token_id = 4
        self.cls_token_id = 2
        self.sep_token_id = 3
    
    def encode(self, text):
        tokens = text.split()
        ids = [self.cls_token_id]
        for t in tokens:
            ids.append(self.token2id.get(t, 1))
        ids.append(self.sep_token_id)
        return ids
    
    def decode(self, ids):
        return ' '.join([self.id2token.get(id, '[UNK]') for id in ids])

tokenizer = SimpleTokenizer()
print(f"✅ 토크나이저 준비 완료 (어휘 크기: {tokenizer.vocab_size})")

## 1. 기본 마스킹 vs Whole Word Masking (WWM) 🆚

### 기본 마스킹의 문제점

서브워드 토크나이저(BPE, WordPiece 등)를 사용하면 하나의 단어가 여러 토큰으로 분리됩니다.

```
원본: "코딩테스트"
서브워드: ["코딩", "##테스트"]  (##은 앞 단어의 일부라는 표시)

기본 마스킹: ["코딩", "[MASK]"]  → '테스트'만 가림 → '코딩'만 보면 너무 쉽게 맞출 수 있음!
WWM:         ["[MASK]", "[MASK]"]  → 단어 전체를 가림 → 문맥에서 추론해야 함!
```

### Whole Word Masking (WWM)
- 서브워드 단위가 아닌 **단어 단위**로 마스킹
- 한 단어의 일부가 마스킹되면 나머지도 모두 마스킹
- 더 어려운 과제 → 더 좋은 표현 학습

In [ ]:
# ============================================================
# 기본 마스킹 구현 (복습)

In [ ]:
def basic_masking(input_ids, tokenizer, mlm_prob=0.15):
    """
    기본 MLM 마스킹: 토큰 단위로 독립적으로 마스킹
    """
    masked_ids = input_ids.copy()
    labels = [-100] * len(input_ids)
    special_ids = set(tokenizer.special_tokens.values())
    
    maskable = [i for i, id in enumerate(input_ids) if id not in special_ids]
    num_mask = max(1, int(len(maskable) * mlm_prob))
    mask_indices = random.sample(maskable, min(num_mask, len(maskable)))
    
    for idx in mask_indices:
        labels[idx] = input_ids[idx]
        r = random.random()
        if r < 0.8:
            masked_ids[idx] = tokenizer.mask_token_id
        elif r < 0.9:
            masked_ids[idx] = random.randint(len(tokenizer.special_tokens), tokenizer.vocab_size - 1)
    
    return masked_ids, labels

print("✅ 기본 마스킹 함수 준비 완료")

In [ ]:
# ============================================================
# Whole Word Masking (WWM) 구현

In [ ]:
def whole_word_masking(input_ids, tokenizer, mlm_prob=0.15):
    """
    Whole Word Masking: 단어 단위로 마스킹
    
    서브워드 토큰(##으로 시작)은 앞 토큰과 함께 마스킹됩니다.
    즉, 하나의 단어에 속하는 모든 서브워드를 함께 마스킹합니다.
    
    Args:
        input_ids: 토큰 ID 리스트
        tokenizer: 토크나이저
        mlm_prob: 마스킹 확률
    
    Returns:
        masked_ids: 마스킹된 ID 리스트
        labels: 정답 레이블
    """
    masked_ids = input_ids.copy()
    labels = [-100] * len(input_ids)
    special_ids = set(tokenizer.special_tokens.values())
    
    # Step 1: 단어 그룹 만들기
    # 서브워드(##으로 시작)를 앞 토큰과 묶기
    word_groups = []   # [[idx1], [idx2, idx3], [idx4], ...]
    current_group = []
    
    for i, token_id in enumerate(input_ids):
        if token_id in special_ids:
            if current_group:
                word_groups.append(current_group)
                current_group = []
            continue
        
        token_text = tokenizer.id2token.get(token_id, '')
        
        if token_text.startswith('##'):
            # 서브워드 → 이전 그룹에 추가
            if current_group:
                current_group.append(i)
            else:
                current_group = [i]
        else:
            # 새 단어 시작
            if current_group:
                word_groups.append(current_group)
            current_group = [i]
    
    if current_group:
        word_groups.append(current_group)
    
    # Step 2: 단어 단위로 마스킹 대상 선택
    num_to_mask = max(1, int(len(word_groups) * mlm_prob))
    mask_groups = random.sample(word_groups, min(num_to_mask, len(word_groups)))
    
    # Step 3: 선택된 단어 그룹의 모든 토큰 마스킹
    for group in mask_groups:
        r = random.random()
        for idx in group:
            labels[idx] = input_ids[idx]
            if r < 0.8:
                masked_ids[idx] = tokenizer.mask_token_id
            elif r < 0.9:
                masked_ids[idx] = random.randint(
                    len(tokenizer.special_tokens), tokenizer.vocab_size - 1
                )
    
    return masked_ids, labels

print("✅ Whole Word Masking 함수 구현 완료!")

In [ ]:
# ============================================================
# 기본 마스킹 vs WWM 비교

In [ ]:
# 서브워드가 포함된 문장 시뮬레이션
# 실제로는 토크나이저가 자동으로 분리하지만, 여기서는 수동으로 시뮬레이션
test_sentence = "나는 고양이 를 좋아한다"
input_ids = tokenizer.encode(test_sentence)

print("📊 기본 마스킹 vs Whole Word Masking 비교")
print("=" * 60)
print(f"원본: {tokenizer.decode(input_ids)}\n")

random.seed(42)
print("--- 기본 마스킹 (5회) ---")
for i in range(5):
    m, l = basic_masking(input_ids, tokenizer, mlm_prob=0.3)
    print(f"  {i+1}: {tokenizer.decode(m)}")

random.seed(42)
print("\n--- Whole Word Masking (5회) ---")
for i in range(5):
    m, l = whole_word_masking(input_ids, tokenizer, mlm_prob=0.3)
    print(f"  {i+1}: {tokenizer.decode(m)}")

print("\n💡 WWM에서는 관련 서브워드가 함께 마스킹되는 것을 확인!")

## 2. Dynamic Masking vs Static Masking 🔄

### Static Masking (정적 마스킹)
- 학습 전에 마스킹을 미리 수행하고 저장
- 매 에폭에서 같은 마스킹 패턴 사용
- 원래 BERT에서 사용 (10개의 다른 마스킹 버전 준비)

### Dynamic Masking (동적 마스킹)
- 매 배치마다 새로운 마스킹을 실시간으로 생성
- RoBERTa에서 제안 (BERT보다 개선된 모델)
- 같은 데이터라도 매번 다른 토큰이 마스킹됨 → 더 다양한 학습

In [ ]:
# ============================================================
# Static vs Dynamic Masking 비교

In [ ]:
print("📊 Static vs Dynamic Masking 비교")
print("=" * 60)

test_ids = tokenizer.encode("나는 고양이 를 좋아한다")

# Static Masking: 한 번 마스킹하고 여러 에폭에 재사용
print("\n🔒 Static Masking (매 에폭 동일한 마스킹):")
random.seed(42)
static_masked, static_labels = basic_masking(test_ids, tokenizer, 0.3)
static_text = tokenizer.decode(static_masked)

for epoch in range(5):
    print(f"  Epoch {epoch+1}: {static_text}  (항상 동일)")

# Dynamic Masking: 매 에폭마다 새로운 마스킹
print("\n🔄 Dynamic Masking (매 에폭 다른 마스킹):")
for epoch in range(5):
    dynamic_masked, _ = basic_masking(test_ids, tokenizer, 0.3)
    print(f"  Epoch {epoch+1}: {tokenizer.decode(dynamic_masked)}  (매번 다름!)")

print("\n💡 Dynamic Masking이 더 다양한 학습 기회를 제공합니다!")
print("   RoBERTa 논문: Dynamic Masking이 Static Masking보다 좋거나 비슷한 성능")

## 3. 마스킹 비율에 따른 효과 실험 📈

BERT는 15%를 마스킹하지만, 다른 비율은 어떨까요?

- **너무 적게 (5%)**: 모델이 충분히 학습하지 못함 (비효율적)
- **적당히 (15%)**: BERT의 기본 설정 (검증된 비율)
- **많이 (30-40%)**: 과제가 너무 어려워질 수 있음
- **너무 많이 (50%+)**: 문맥이 부족하여 제대로 추론 불가

In [ ]:
# ============================================================
# 마스킹 비율별 효과 시각화

In [ ]:
test_sentence = "나는 어제 도서관 에서 강아지 와 고양이 를 본다"
test_ids = tokenizer.encode(test_sentence)

mlm_probs = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

print("📊 마스킹 비율별 결과 비교")
print("=" * 60)
print(f"원본: {tokenizer.decode(test_ids)}")
print(f"총 토큰 수: {len(test_ids)} (특수 토큰 포함)\n")

for prob in mlm_probs:
    # 10번 실행하여 평균 마스킹 토큰 수 계산
    mask_counts = []
    for _ in range(100):
        m, l = basic_masking(test_ids, tokenizer, prob)
        mask_count = sum(1 for x in m if x == tokenizer.mask_token_id)
        mask_counts.append(mask_count)
    
    avg_masks = np.mean(mask_counts)
    # 예시 하나 출력
    example_m, _ = basic_masking(test_ids, tokenizer, prob)
    print(f"  {int(prob*100):2d}%: {tokenizer.decode(example_m)}")
    print(f"       평균 마스킹 수: {avg_masks:.1f}개\n")

# 시각화
fig, ax = plt.subplots(figsize=(10, 5))

mask_counts_by_prob = []
for prob in mlm_probs:
    counts = []
    for _ in range(200):
        m, _ = basic_masking(test_ids, tokenizer, prob)
        counts.append(sum(1 for x in m if x == tokenizer.mask_token_id))
    mask_counts_by_prob.append(counts)

ax.boxplot(mask_counts_by_prob, labels=[f'{int(p*100)}%' for p in mlm_probs])
ax.set_xlabel('마스킹 비율', fontsize=12)
ax.set_ylabel('마스킹된 토큰 수', fontsize=12)
ax.set_title('마스킹 비율별 마스킹 토큰 수 분포', fontsize=14, fontweight='bold')
ax.axhline(y=len(test_ids)*0.15, color='r', linestyle='--', alpha=0.5, label='BERT 기본 (15%)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. SpanBERT 스타일 연속 마스킹 (Span Masking)

SpanBERT는 개별 토큰 대신 **연속된 토큰 구간(span)**을 마스킹합니다.

```
기본 BERT: "나는 [MASK] 에서 [MASK] 를 [MASK]"
SpanBERT:  "나는 [MASK] [MASK] [MASK] 를 본다"
           (연속된 구간을 한 번에 마스킹)
```

### 왜 연속 마스킹?
- 단일 토큰 예측보다 더 어려운 과제 → 더 깊은 이해 필요
- 문장 구조와 의미를 더 잘 학습
- span 길이는 기하 분포에서 샘플링 (평균 3.8 토큰)

In [ ]:
# ============================================================
# Span Masking 구현

In [ ]:
def span_masking(input_ids, tokenizer, mlm_prob=0.15, max_span_length=5):
    """
    SpanBERT 스타일 연속 마스킹
    
    연속된 토큰 구간(span)을 마스킹합니다.
    span 길이는 기하 분포에서 샘플링합니다.
    
    Args:
        input_ids: 토큰 ID 리스트
        tokenizer: 토크나이저
        mlm_prob: 마스킹 확률 (전체 토큰 대비)
        max_span_length: 최대 span 길이
    
    Returns:
        masked_ids, labels
    """
    masked_ids = input_ids.copy()
    labels = [-100] * len(input_ids)
    special_ids = set(tokenizer.special_tokens.values())
    
    # 마스킹 가능한 위치
    maskable = [i for i, id in enumerate(input_ids) if id not in special_ids]
    
    # 목표 마스킹 토큰 수
    target_num_masked = max(1, int(len(maskable) * mlm_prob))
    
    masked_set = set()  # 이미 마스킹된 위치
    
    while len(masked_set) < target_num_masked:
        # Span 길이 샘플링 (기하 분포, 평균 약 3)
        span_length = min(
            np.random.geometric(p=0.3),  # 기하 분포
            max_span_length,
            target_num_masked - len(masked_set)  # 남은 목표 수
        )
        
        # Span 시작 위치 랜덤 선택 (마스킹 가능한 위치에서)
        available = [i for i in maskable if i not in masked_set]
        if not available:
            break
        
        start = random.choice(available)
        
        # Span에 포함될 위치 결정
        span_indices = []
        for offset in range(span_length):
            pos = start + offset
            if pos in maskable and pos not in masked_set:
                if pos < len(input_ids) and input_ids[pos] not in special_ids:
                    span_indices.append(pos)
                else:
                    break
            else:
                break
        
        # Span 마스킹 적용
        r = random.random()
        for idx in span_indices:
            labels[idx] = input_ids[idx]
            masked_set.add(idx)
            
            if r < 0.8:
                masked_ids[idx] = tokenizer.mask_token_id
            elif r < 0.9:
                masked_ids[idx] = random.randint(
                    len(tokenizer.special_tokens), tokenizer.vocab_size - 1
                )
    
    return masked_ids, labels

print("✅ Span Masking 함수 구현 완료!")

In [ ]:
# ============================================================
# 기본 마스킹 vs Span Masking 비교

In [ ]:
test_sentence = "나는 어제 도서관 에서 강아지 와 고양이 를 본다"
test_ids = tokenizer.encode(test_sentence)

print("📊 기본 마스킹 vs Span Masking 비교")
print("=" * 60)
print(f"원본: {tokenizer.decode(test_ids)}\n")

print("--- 기본 마스킹 (개별 토큰) ---")
for i in range(5):
    m, _ = basic_masking(test_ids, tokenizer, 0.3)
    print(f"  {tokenizer.decode(m)}")

print("\n--- Span Masking (연속 구간) ---")
for i in range(5):
    m, _ = span_masking(test_ids, tokenizer, 0.3)
    print(f"  {tokenizer.decode(m)}")

print("\n💡 Span Masking에서 [MASK]가 연속으로 나타나는 것을 확인!")
print("   연속 구간을 예측하려면 더 깊은 문맥 이해가 필요합니다.")

## 5. HuggingFace Transformers 활용 (실전) 🤗

실제로는 HuggingFace의 라이브러리를 사용하면 매우 간단합니다.

아래는 실전에서 사용하는 코드의 예시입니다.

(⚠️ 실행하려면 `pip install transformers`가 필요합니다)

In [ ]:
# ============================================================
# HuggingFace 스타일 코드 (참고용)

In [ ]:
# 아래 코드는 HuggingFace transformers가 설치된 환경에서 실행 가능합니다.
# 설치: pip install transformers

huggingface_code = '''

In [ ]:
# 실전에서의 BERT MLM 학습 코드 (HuggingFace 활용)

In [ ]:
from transformers import (
    BertTokenizer, 
    BertForMaskedLM, 
    DataCollatorForLanguageModeling,
    Trainer, 
    TrainingArguments
)
from datasets import load_dataset

# 1. 토크나이저 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 2. 모델 로드  
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# 3. Data Collator (마스킹 자동 수행!)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,           # MLM 마스킹 활성화
    mlm_probability=0.15 # 15% 마스킹
)

# 4. 학습 설정
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
)

# 5. Trainer로 학습
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()
'''

print("📝 HuggingFace 실전 코드 (참고용)")
print("=" * 50)
print(huggingface_code)
print("💡 이 6개의 노트북에서 배운 개념들이 실전에서는 이렇게 간단히 사용됩니다!")
print("   하지만 내부 동작 원리를 이해하는 것이 중요합니다.")

## 📝 전체 시리즈 핵심 정리

### 🔄 6개 노트북에서 배운 내용:

| 노트북 | 주제 | 핵심 개념 |
|--------|------|----------|
| **1** | Scaled Dot-Product Attention | Q, K, V, 스케일링, Softmax |
| **2** | Attention 심화 실습 | Padding Mask, Self/Cross Attention |
| **3** | Causal Attention | 하삼각 마스크, 자동 회귀 생성 |
| **4** | Multi-Head Attention | Head 분리/결합, 다양한 관점 학습 |
| **5** | BERT MLM Data Collator | 마스킹 전략, 토크나이저, 배치 처리 |
| **6** | MLM 마스킹 심화 | WWM, Dynamic Masking, Span Masking |

### 🏗️ Transformer 아키텍처에서의 위치:

```
Transformer
├── Encoder (BERT 스타일)
│   ├── Self-Attention (Notebook 1, 2, 4)
│   ├── Padding Mask (Notebook 2)
│   └── MLM 사전학습 (Notebook 5, 6)
│
└── Decoder (GPT 스타일)
    ├── Masked Self-Attention (Notebook 3, 4)
    ├── Cross-Attention (Notebook 2)
    └── Causal Mask (Notebook 3)
```

### 🚀 다음 단계 제안
1. 전체 Transformer 모델 구현 (Encoder + Decoder)
2. Position Encoding 구현
3. Feed-Forward Network, Layer Normalization 추가
4. 실제 데이터로 학습 및 평가
5. HuggingFace Transformers 활용한 실전 프로젝트

---

**수고하셨습니다! 🎉**

## AI Developed Version 3
Source: 068_MLM_Masking_Implementation.ipynb

# 📘 실습 068: Masked Language Model 마스킹 심화 구현

## 🎯 이번 실습에서 다룰 것

067에서 기본 MLM 마스킹을 배웠습니다. 이번에는 더 깊이 들어갑니다:

1. **기본 토큰 마스킹** - 067 복습
2. **Whole Word Masking (WWM)** - 서브워드가 아닌 전체 단어 마스킹
3. **스팬(Span) 마스킹** - 연속된 여러 토큰을 함께 마스킹
4. **작은 BERT MLM 모델 직접 학습** - 실제 학습 루프 구현
5. **학습 전후 성능 비교**

---

### 🤔 왜 다양한 마스킹 전략이 필요한가?

```
기본 마스킹:
  원본: "I am studying transformer models"
  마스크: "I [MASK] studying [MASK] [MASK]"
  → 'transformer'의 일부(sub-word)만 마스킹되면 전체를 추론하기 쉬움

Whole Word Masking:
  원본: "I am studying transformer models"
  마스크: "I am [MASK] [MASK] [MASK] [MASK] models"
  ("transformer" = ["transform", "##er"] → 두 토큰 모두 마스킹)
  → 더 어렵고 더 깊은 이해 필요

Span Masking:
  원본: "I am studying transformer models"
  마스크: "I am [MASK] [MASK] [MASK] models"
  (연속 3개 토큰을 하나의 스팬으로 마스킹)
  → 긴 의존성 학습에 효과적
```

In [ ]:
# !pip install transformers datasets torch

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import math
from transformers import BertTokenizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 라이브러리 로드 완료 | 디바이스: {device}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("✅ BERT 토크나이저 로드 완료")

In [ ]:
# ============================================================
# 🎭 Step 1: 기본 토큰 마스킹 vs Whole Word Masking 비교

In [ ]:
def apply_basic_masking(token_ids, tokenizer, mask_prob=0.15):
    """
    기본 BERT 마스킹 (서브워드 단위)
    각 토큰을 독립적으로 15% 확률로 마스킹
    """
    token_ids = list(token_ids)
    labels = [-100] * len(token_ids)  # 기본값: -100 (무시)
    
    for i, token_id in enumerate(token_ids):
        # 특수 토큰은 마스킹하지 않음
        if token_id in [tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id]:
            continue
        
        if random.random() < mask_prob:
            labels[i] = token_id  # 정답 저장
            
            r = random.random()
            if r < 0.8:
                token_ids[i] = tokenizer.mask_token_id   # 80%: [MASK]
            elif r < 0.9:
                token_ids[i] = random.randint(0, tokenizer.vocab_size-1)  # 10%: 랜덤
            # else: 10% 원본 유지
    
    return token_ids, labels


def apply_whole_word_masking(tokens, token_ids, tokenizer, mask_prob=0.15):
    """
    Whole Word Masking (WWM)
    서브워드 토큰을 원래 단어로 그룹화한 후, 단어 단위로 마스킹
    
    예: "transformer" → ["transform", "##er"]
    → 두 토큰을 함께 마스킹하거나 함께 유지
    """
    token_ids = list(token_ids)
    labels = [-100] * len(token_ids)
    
    # 단어 그룹 찾기: ##으로 시작하는 토큰은 앞 단어의 일부
    word_groups = []
    current_group = []
    
    for i, token in enumerate(tokens):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            if current_group:
                word_groups.append(current_group)
                current_group = []
            word_groups.append([i])  # 특수 토큰은 단독 그룹
        elif token.startswith('##'):
            # 서브워드: 이전 그룹에 추가
            current_group.append(i)
        else:
            # 새 단어 시작
            if current_group:
                word_groups.append(current_group)
            current_group = [i]
    
    if current_group:
        word_groups.append(current_group)
    
    # 단어 단위로 마스킹 결정
    for group in word_groups:
        first_token = tokens[group[0]] if group[0] < len(tokens) else ''
        
        # 특수 토큰 그룹은 마스킹 안 함
        if first_token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        
        if random.random() < mask_prob:
            # 단어의 모든 서브워드 토큰을 함께 마스킹!
            for i in group:
                if i < len(token_ids):
                    labels[i] = token_ids[i]  # 정답
                    r = random.random()
                    if r < 0.8:
                        token_ids[i] = tokenizer.mask_token_id
                    elif r < 0.9:
                        token_ids[i] = random.randint(0, tokenizer.vocab_size-1)
    
    return token_ids, labels


# 비교 예시
sample = "The transformer architecture revolutionizes natural language processing."
tokens = tokenizer.tokenize(sample)
token_ids = tokenizer.encode(sample)

print("=" * 70)
print("기본 마스킹 vs Whole Word Masking 비교")
print("=" * 70)
print(f"\n원본 문장: {sample}")
print(f"\n토크나이징 결과:")
print(f"  {tokens}")
print()

# 서브워드 확인
subwords = [(i, t) for i, t in enumerate(tokens) if t.startswith('##')]
if subwords:
    print("서브워드 토큰 (## 표시):")
    for idx, tok in subwords:
        print(f"  위치 {idx}: '{tok}' (앞 토큰 '{tokens[idx-1]}'의 일부)")

print()

# 여러 번 실행해서 비교
random.seed(10)
basic_ids, basic_labels = apply_basic_masking(token_ids, tokenizer)
random.seed(10)
wwm_ids, wwm_labels = apply_whole_word_masking(
    ['[CLS]'] + tokens + ['[SEP]'], token_ids, tokenizer
)

print(f"{'토큰':<20} {'기본 마스킹':<20} {'WWM':<20}")
print("-" * 60)

all_tokens = ['[CLS]'] + tokens + ['[SEP]']
for i, (orig, basic, wwm) in enumerate(zip(all_tokens[:15], 
                                            tokenizer.convert_ids_to_tokens(basic_ids[:15]),
                                            tokenizer.convert_ids_to_tokens(wwm_ids[:15]))):
    basic_mark = '← 마스킹!' if basic == '[MASK]' else ''
    wwm_mark = '← 마스킹!' if wwm == '[MASK]' else ''
    print(f"{orig:<20} {basic:<20} {wwm:<20}")

In [ ]:
# ============================================================
# 🔀 Step 2: Span Masking 구현

In [ ]:
#
# Span Masking: T5 모델에서 사용, SpanBERT 등에서도 활용
# 연속된 여러 토큰을 하나의 '스팬'으로 묶어서 함께 마스킹
# → 더 긴 문맥 의존성 학습에 효과적

def apply_span_masking(token_ids, tokenizer, mask_ratio=0.15, mean_span_length=3):
    """
    Span Masking 구현
    
    SpanBERT 논문 (Joshi et al., 2020)에서 제안
    
    Args:
        token_ids:        토큰 ID 리스트
        tokenizer:        토크나이저
        mask_ratio:       전체 토큰 중 마스킹할 비율 (기본: 0.15)
        mean_span_length: 평균 스팬 길이 (기본: 3)
    
    스팬 길이는 기하분포(Geometric Distribution)를 따름:
    → 짧은 스팬이 많고, 긴 스팬은 드물게
    """
    token_ids = list(token_ids)
    labels = [-100] * len(token_ids)
    
    # 마스킹 가능한 위치 (특수 토큰 제외)
    maskable = [
        i for i, tid in enumerate(token_ids)
        if tid not in [tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id]
    ]
    
    # 마스킹할 총 토큰 수
    num_to_mask = int(len(maskable) * mask_ratio)
    
    masked_count = 0
    attempts = 0
    
    while masked_count < num_to_mask and attempts < 100:
        attempts += 1
        
        # 기하분포로 스팬 길이 결정
        # p = 1/mean_span_length (기하분포 매개변수)
        # 평균 = mean_span_length
        p = 1.0 / mean_span_length
        span_length = 1
        while random.random() > p and span_length < 10:  # 최대 10
            span_length += 1
        
        # 랜덤 시작 위치 선택
        if not maskable:
            break
        start_pos = random.choice(maskable)
        
        # 스팬 마스킹 적용
        for offset in range(span_length):
            pos = start_pos + offset
            if pos < len(token_ids) and pos in maskable:
                labels[pos] = token_ids[pos]
                token_ids[pos] = tokenizer.mask_token_id
                masked_count += 1
                if pos in maskable:
                    maskable.remove(pos)
    
    return token_ids, labels


# 3가지 마스킹 전략 시각화
sample2 = "BERT learns deep representations of language through masked prediction tasks."
tokens2 = tokenizer.tokenize(sample2)
token_ids2 = tokenizer.encode(sample2)
all_tokens2 = ['[CLS]'] + tokens2 + ['[SEP]']

random.seed(5)
basic2, basic_labels2 = apply_basic_masking(token_ids2, tokenizer)
random.seed(5)
wwm2, wwm_labels2 = apply_whole_word_masking(all_tokens2, token_ids2, tokenizer)
random.seed(5)
span2, span_labels2 = apply_span_masking(token_ids2, tokenizer)

# 시각화
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

methods = ['기본 마스킹', 'Whole Word Masking', 'Span Masking']
masked_lists = [basic2, wwm2, span2]
colors_map = {'[MASK]': '#e74c3c', 'original': '#3498db', '[CLS]': '#95a5a6', '[SEP]': '#95a5a6'}

num_show = min(15, len(all_tokens2))

for ax_idx, (ax, method, masked) in enumerate(zip(axes, methods, masked_lists)):
    colors = []
    display_tokens = []
    
    for i in range(num_show):
        orig_id = token_ids2[i] if i < len(token_ids2) else 0
        mask_id = masked[i] if i < len(masked) else 0
        orig_tok = all_tokens2[i] if i < len(all_tokens2) else ''
        
        if orig_tok in ['[CLS]', '[SEP]']:
            colors.append('#95a5a6')
            display_tokens.append(orig_tok)
        elif mask_id == tokenizer.mask_token_id:
            colors.append('#e74c3c')
            display_tokens.append('[MASK]')
        else:
            colors.append('#3498db')
            display_tokens.append(orig_tok)
    
    for i, (tok, color) in enumerate(zip(display_tokens, colors)):
        rect = mpatches.FancyBboxPatch(
            (i * 0.95, 0.1), 0.9, 0.8,
            boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='white', alpha=0.85
        )
        ax.add_patch(rect)
        ax.text(i * 0.95 + 0.45, 0.5, tok,
                ha='center', va='center', fontsize=7,
                color='white', fontweight='bold')
    
    # 마스킹 비율
    mask_count = sum(1 for id in masked[:num_show] if id == tokenizer.mask_token_id)
    real_count = num_show - 2  # [CLS], [SEP] 제외
    ax.set_xlim(-0.1, num_show * 0.95 + 0.1)
    ax.set_ylim(0, 1)
    ax.set_title(f'{method} (마스킹 비율: {mask_count/real_count*100:.1f}%)', fontsize=11, fontweight='bold')
    ax.axis('off')

# 범례
legend_elements = [
    mpatches.Patch(facecolor='#e74c3c', label='[MASK] (마스킹됨)'),
    mpatches.Patch(facecolor='#3498db', label='원본 유지'),
    mpatches.Patch(facecolor='#95a5a6', label='특수 토큰'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10)
plt.suptitle(f'마스킹 전략 비교\n원본: "{sample2[:50]}..."', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig('masking_strategies_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 🏗️ Step 3: 작은 BERT 스타일 MLM 모델 구현

In [ ]:
#
# 실제 BERT는 너무 크지만, 같은 구조의 미니 버전을 만들어봅니다.
# (실제 학습이 가능한 크기)

class MultiHeadSelfAttention(nn.Module):
    """멀티헤드 셀프 어텐션 (양방향, 마스크 없음)"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(x)


class MiniBertBlock(nn.Module):
    """BERT 스타일 인코더 블록"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),  # BERT는 GELU 사용 (GPT 스타일)
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-LN 방식 (학습 안정성이 더 좋음)
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class MiniBERT(nn.Module):
    """
    미니 BERT 모델 (MLM 사전학습용)
    
    구조:
    토큰 임베딩 + 위치 임베딩 → N × BERT 블록 → MLM 헤드 → 토큰 예측
    
    실제 BERT-base:
    - d_model=768, num_heads=12, d_ff=3072, num_layers=12
    - 파라미터 수: ~110M
    
    우리 Mini BERT:
    - d_model=128, num_heads=4, d_ff=256, num_layers=2
    - 파라미터 수: ~수백만 (학습 가능한 크기)
    """
    
    def __init__(self, vocab_size, d_model=128, num_heads=4,
                 d_ff=256, num_layers=2, max_len=128, dropout=0.1):
        super().__init__()
        
        # 임베딩 레이어
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.segment_emb = nn.Embedding(2, d_model)  # 문장 A/B 구분 (NSP용)
        self.emb_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # BERT 인코더 블록들
        self.layers = nn.ModuleList([
            MiniBertBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # MLM 헤드: 각 위치에서 원래 토큰을 예측
        # Linear → GELU → LayerNorm → Linear (vocab_size로)
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, vocab_size)
        )
        
        # 가중치 초기화
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """BERT 스타일 가중치 초기화"""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            module.bias.data.zero_()
    
    def forward(self, input_ids, attention_mask=None, segment_ids=None):
        """
        Args:
            input_ids:      (batch, seq_len) 마스킹된 토큰 IDs
            attention_mask: (batch, seq_len) 패딩 위치 표시 (1=실제, 0=패딩)
            segment_ids:    (batch, seq_len) 문장 A=0, B=1
        
        Returns:
            logits: (batch, seq_len, vocab_size) 각 위치의 토큰 예측 확률
        """
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        
        if segment_ids is None:
            segment_ids = torch.zeros_like(input_ids)
        
        # 세 가지 임베딩 합산
        x = (self.token_emb(input_ids) +
             self.pos_emb(positions) +
             self.segment_emb(segment_ids))
        x = self.dropout(self.emb_norm(x))
        
        # Attention mask 변환 (1=실제 → 1=허용, 0=패딩 → 마스킹)
        if attention_mask is not None:
            # (B, T) → (B, 1, 1, T) → 브로드캐스팅
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
        else:
            mask = None
        
        # BERT 레이어 통과
        for layer in self.layers:
            x = layer(x, mask)
        
        # MLM 헤드: 각 위치에서 원래 토큰 예측
        logits = self.mlm_head(x)
        
        return logits


# 모델 생성 및 정보 출력
mini_bert = MiniBERT(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    num_heads=4,
    d_ff=256,
    num_layers=2,
    max_len=64
).to(device)

total_params = sum(p.numel() for p in mini_bert.parameters())
trainable_params = sum(p.numel() for p in mini_bert.parameters() if p.requires_grad)

print("=" * 60)
print("Mini BERT 모델 정보")
print("=" * 60)
print(f"총 파라미터: {total_params:,}")
print(f"학습 가능 파라미터: {trainable_params:,}")
print(f"\n비교:")
print(f"  BERT-base:  ~110,000,000 (110M)")
print(f"  BERT-large: ~340,000,000 (340M)")
print(f"  Mini BERT:  {total_params:>14,} ({total_params/1e6:.1f}M)")

In [ ]:
# ============================================================
# 🏋️ Step 4: Mini BERT MLM 학습

In [ ]:
from torch.utils.data import DataLoader, Dataset

# 학습 데이터
train_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is changing the world rapidly.",
    "Deep learning uses neural networks to learn from data.",
    "The transformer model revolutionized natural language processing.",
    "BERT learns contextual word representations.",
    "Machine learning algorithms can identify complex patterns.",
    "Python is the most popular language for data science tasks.",
    "Natural language understanding requires deep contextual knowledge.",
    "Self-attention mechanisms allow models to focus on relevant parts.",
    "Pre-training on large corpora improves downstream task performance.",
    "The encoder processes input sequences to extract meaningful features.",
    "Positional encoding helps the model understand sequence order.",
    "Multi-head attention captures different aspects of relationships.",
    "Layer normalization stabilizes the training process significantly.",
    "Residual connections help gradients flow through deep networks.",
    "Fine-tuning adapts pre-trained models to specific downstream tasks.",
]

class MLMDataset(Dataset):
    """MLM 학습을 위한 데이터셋"""
    
    def __init__(self, texts, tokenizer, max_length=64, mask_prob=0.15):
        self.mask_prob = mask_prob
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # 모든 텍스트 인코딩
        self.examples = []
        for text in texts:
            ids = tokenizer.encode(
                text,
                max_length=max_length,
                truncation=True,
                padding='max_length'
            )
            self.examples.append(ids)
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        token_ids = self.examples[idx]
        input_ids, labels = self._mask_tokens(token_ids)
        attn_mask = [1 if tid != self.tokenizer.pad_token_id else 0 for tid in token_ids]
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
            'attention_mask': torch.tensor(attn_mask, dtype=torch.long)
        }
    
    def _mask_tokens(self, token_ids):
        input_ids = list(token_ids)
        labels = [-100] * len(token_ids)
        
        for i, tid in enumerate(token_ids):
            if tid in [self.tokenizer.cls_token_id, self.tokenizer.sep_token_id, 
                       self.tokenizer.pad_token_id]:
                continue
            
            if random.random() < self.mask_prob:
                labels[i] = tid
                r = random.random()
                if r < 0.8:
                    input_ids[i] = self.tokenizer.mask_token_id
                elif r < 0.9:
                    input_ids[i] = random.randint(0, self.tokenizer.vocab_size - 1)
        
        return input_ids, labels


# 데이터셋 및 DataLoader
dataset = MLMDataset(train_texts, tokenizer, max_length=32)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# 옵티마이저
optimizer = torch.optim.AdamW(mini_bert.parameters(), lr=1e-3, weight_decay=0.01)
# 학습률 스케줄러 (처음에 올리고 서서히 감소)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# 학습 루프
num_epochs = 30
losses = []

mini_bert.train()
print("학습 시작...")

for epoch in range(num_epochs):
    epoch_loss = 0
    num_batches = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        
        # 순전파
        logits = mini_bert(input_ids, attn_mask)
        # logits: (batch, seq, vocab_size)
        # labels: (batch, seq)  -100은 무시
        
        # Loss 계산: CrossEntropy, ignore_index=-100
        loss = F.cross_entropy(
            logits.view(-1, tokenizer.vocab_size),  # (batch*seq, vocab)
            labels.view(-1),                         # (batch*seq,)
            ignore_index=-100
        )
        
        # 역전파
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mini_bert.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    scheduler.step()
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

print("\n✅ 학습 완료!")

In [ ]:
# ============================================================
# 📊 Step 5: 학습 결과 분석

In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 학습 손실
axes[0].plot(losses, color='#3498db', linewidth=2)
axes[0].fill_between(range(len(losses)), losses, alpha=0.2, color='#3498db')
axes[0].set_title('Mini BERT MLM 학습 손실', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)
axes[0].annotate(f'최종 Loss: {losses[-1]:.4f}', 
                  xy=(len(losses)-1, losses[-1]),
                  xytext=(len(losses)*0.6, losses[0]*0.8),
                  arrowprops=dict(arrowstyle='->', color='red'),
                  fontsize=10, color='red')

# 마스킹된 토큰 예측 시각화
mini_bert.eval()
test_sentence = "The transformer model learns contextual representations."
test_ids = tokenizer.encode(test_sentence, max_length=32, padding='max_length', truncation=True)
test_tokens = tokenizer.convert_ids_to_tokens(test_ids)

# 수동 마스킹 ("contextual"과 "representations" 마스킹)
masked_test_ids = list(test_ids)
mask_positions = []
for i, tok in enumerate(test_tokens):
    if tok in ['contextual', 'representations', 'represent', '##ations']:
        masked_test_ids[i] = tokenizer.mask_token_id
        mask_positions.append(i)

# 예측
with torch.no_grad():
    input_tensor = torch.tensor([masked_test_ids], device=device)
    attn = torch.tensor([[1 if t != tokenizer.pad_token_id else 0 for t in test_ids]], device=device)
    logits = mini_bert(input_tensor, attn)
    predictions = logits[0].argmax(dim=-1)

print("=" * 60)
print("마스킹된 토큰 예측 결과")
print("=" * 60)
print(f"\n입력: {test_sentence}")
print(f"\n마스킹된 위치와 예측:")

for pos in mask_positions:
    orig_tok = test_tokens[pos]
    pred_id = predictions[pos].item()
    pred_tok = tokenizer.convert_ids_to_tokens([pred_id])[0]
    # 상위 3개 예측
    top3_ids = logits[0, pos].topk(3).indices.tolist()
    top3_toks = tokenizer.convert_ids_to_tokens(top3_ids)
    print(f"  위치 {pos}: 원본='{orig_tok}', 예측='{pred_tok}'")
    print(f"    상위 3개: {top3_toks}")

print("\n(학습 데이터가 적어 완벽하지 않지만 구조는 동일!)")

# 예측 분포 시각화
if mask_positions:
    probs = F.softmax(logits[0, mask_positions[0]], dim=-1)
    top10 = probs.topk(10)
    top10_toks = tokenizer.convert_ids_to_tokens(top10.indices.tolist())
    top10_vals = top10.values.tolist()
    
    axes[1].barh(range(10), top10_vals[::-1], color='#2ecc71', edgecolor='black', alpha=0.8)
    axes[1].set_yticks(range(10))
    axes[1].set_yticklabels(top10_toks[::-1], fontsize=9)
    axes[1].set_title(f'[MASK] 위치 예측 Top-10\n(위치 {mask_positions[0]})', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('예측 확률')
    axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('mlm_training_results.png', dpi=100, bbox_inches='tight')
plt.show()

## 📋 전체 시리즈 정리

### ✅ 063-068 실습 시리즈에서 배운 것

| 노트북 | 주제 | 핵심 개념 |
|--------|------|-----------|
| **063** | Scaled Dot-Product Attention | Q, K, V, softmax, 스케일링 |
| **064** | Attention 실습 | Cross-Attention, 패딩 마스크, Temperature |
| **065** | Causal Attention | 미래 마스킹, 자기회귀 LM |
| **066** | Multi-Head Attention | 병렬 헤드, Transformer 인코더 |
| **067** | BERT MLM Data Collator | 마스킹 규칙, DataLoader |
| **068** | MLM 심화 구현 | WWM, Span Masking, 실제 학습 |

### 🚀 다음 학습 방향
1. **실제 BERT Fine-tuning** - 감정 분석, QA 등 실제 태스크
2. **GPT 스타일 생성** - Causal LM으로 텍스트 생성
3. **T5, RoBERTa 등** - 다양한 변형 모델 탐구
4. **HuggingFace 활용** - 대규모 사전학습 모델 파인튜닝

---
*이 실습 자료는 Transformer 입문자를 위해 상세한 주석과 함께 제공됩니다.*